# **Building Production-Grade Data Agents on GCP**

## Notebook 1 — Build Your First Enterprise Data Agent

This notebook builds a read-only Gemini + ADK inventory-risk agent over BigQuery.

**Start here:** open this notebook directly from the public GitHub repository in Google Colab. Its first cells clone the repository (if needed), install dependencies, load the participant configuration and initialise Google Cloud.

No terminal, local IDE, separate setup notebook or Gemini API key is required. Authentication uses your Google account through Colab ADC and Vertex AI.


In [ ]:
# Bootstrap: clone repository and install dependencies from inside Colab.
from pathlib import Path
import subprocess, sys, os, json

REPOSITORY_URL = "https://github.com/REPLACE_WITH_GITHUB_OWNER/enterprise-data-agents-gdg-essex.git"
REPOSITORY_REF = "main"
REPO_DIR = Path("/content/enterprise-data-agents-gdg-essex")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
         REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Workshop repository already cloned in this runtime.")

%cd /content/enterprise-data-agents-gdg-essex
%pip install -q -r requirements-agent.txt

# Participant settings: edit these values before running the cell.
# The organiser supplies the source project and dataset identifiers.
PARTICIPANT_CONFIG = {
    "google_cloud": {
        "project_id": "REPLACE_WITH_YOUR_GCP_PROJECT_ID",
        "bigquery_location": "US",
        "dataset_id": "essex_workshop",
    },
    "vertex_ai": {
        "model_id": "gemini-3.8-flash",
        "location": "global",
    },
    "source_data": {
        "project_id": "REPLACE_WITH_WORKSHOP_SOURCE_PROJECT_ID",
        "dataset_id": "REPLACE_WITH_WORKSHOP_SOURCE_DATASET_ID",
    },
    "repository": {
        "url": REPOSITORY_URL,
        "branch_or_tag": REPOSITORY_REF,
    },
}

import yaml
config_path = REPO_DIR / "config" / "workshop_config.yaml"
config_path.write_text(yaml.safe_dump(PARTICIPANT_CONFIG, sort_keys=False), encoding="utf-8")

from setup.workshop_setup import initialise_workshop
from google.colab import auth

workshop = initialise_workshop(config_path=config_path, repo_dir=REPO_DIR)
bq = workshop.bigquery
PROJECT_ID = workshop.project_id
DATASET_ID = workshop.dataset_id
BIGQUERY_LOCATION = workshop.location
MODEL_ID = workshop.model_id
MODEL_LOCATION = workshop.model_location

print("Workshop setup complete.")
print("Project:", PROJECT_ID, "| Dataset:", DATASET_ID)
print("Gemini:", MODEL_ID, "| Vertex AI location:", MODEL_LOCATION)


## Learning objectives

By the end, you should be able to explain the agent runtime, use bounded read-only BigQuery tools, and ground an inventory-risk answer in canonical business definitions.


## 1. The mental model: from question to evidence-backed answer

A useful first agent combines a model, instructions, bounded tools, a session and a runner. Harness Engineering provides the runtime and operational controls around execution.

```text
User question → ADK Runner + Session → Gemini + instructions → Read-only BigQuery tool → Semantic view → Evidence-backed answer
```


In [ ]:
from tools.bigquery_tools import SupplyChainBigQueryTools

supply_chain_tools = SupplyChainBigQueryTools(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    location=BIGQUERY_LOCATION,
)

def get_inventory_risk():
    """Return highest-value inventory safety-stock gaps from the latest snapshot."""
    result = supply_chain_tools.inventory_risk(
        inventory_status=None, category=None, limit=100
    )
    risk_rows = [
        row for row in result["rows"]
        if int(row.get("shortage_units") or 0) > 0
    ]
    return {
        "rows": risk_rows[:20],
        "row_count": len(risk_rows[:20]),
        "total_risk_rows_returned": len(risk_rows),
        "read_only": True,
        "note": "Rows are ordered by inventory value at risk, then shortage units.",
    }

def get_inventory_summary():
    """Return canonical inventory portfolio metrics for the latest snapshot."""
    return supply_chain_tools.inventory_summary()

print("Read-only BigQuery tools ready.")


## Learning objectives

By the end, you should be able to:

- Explain the difference between a chatbot response and an agent that can use tools.
- Describe the basic agent runtime (model, instructions, tools, session and runner).
- Explain **Harness Engineering**: the runtime and operational controls that make agent execution reliable.
- Use a small, allow-listed BigQuery tool surface rather than letting a model write arbitrary SQL.
- Ground an inventory-risk answer in the workshop's business definitions and query evidence.

### The scenario

You are supporting the **Essex Central Distribution Centre** (`WH-ESSEX-01`). The business wants to know which products have the greatest inventory risk and what the cost value of the safety-stock gap is.

The dataset is synthetic and uses GBP. Its snapshot date is `2026-10-01`. The first question is deliberately read-only:

> Which products are most at risk of falling below safety stock, and what is the inventory value at risk?

This notebook does not create purchase orders, change stock, approve spend or contact suppliers.

## 1. The mental model: from question to evidence-backed answer

A useful first agent is not an autonomous system with unrestricted access. It is a model that can reason over a bounded set of permitted tools and return an answer grounded in their results.

<pre>
User question
     |
     v
ADK Runner + Session  <--- Harness Engineering
     |
     v
Gemini model + instructions
     |
     +---- decides a tool is needed
     |             |
     |             v
     |      Read-only BigQuery function
     |             |
     |             v
     |      Governed semantic view
     |             |
     |             v
     |      Structured tool result
     |             |
     +<------------+
     |
     v
Evidence-grounded response
</pre>

**The key pieces**

- **Model:** interprets the request and produces a response or tool call.
- **Instructions/context:** define the role, business language, constraints and answer expectations.
- **Tool:** a narrowly scoped Python function that performs one permitted action.
- **Session:** holds the conversation state for a user interaction.
- **Runner:** executes the agent and coordinates model/tool events.
- **Harness Engineering:** the surrounding runtime, lifecycle, validation, error handling, bounded execution and tracing. It is not the same thing as the Data Harness introduced in Notebook 2.

## 2. Inspect the tool result before giving it to an agent

A good practice is to test a tool independently. This separates data-access errors from agent reasoning errors and makes the evidence visible.

In [ ]:
inventory_evidence = get_inventory_risk()
inventory_summary_evidence = get_inventory_summary()

print("Risk rows returned:", inventory_evidence["row_count"])
print("Read-only:", inventory_evidence["read_only"])
print("\nHighest-risk products:")
for row in inventory_evidence["rows"][:8]:
    print(
        f"{row['product_id']} | {row['product_name']} | "
        f"shortage={row['shortage_units']} units | "
        f"value at risk=£{float(row['inventory_value_at_risk_gbp']):,.2f}"
    )

print("\nCanonical portfolio summary:")
for row in inventory_summary_evidence["rows"]:
    print(row)

## 3. Build the first ADK agent

Now we wrap the tested Python functions as ADK tools. The agent has a clear job, a small toolset and explicit boundaries.

Notice the separation:

- The **tool** retrieves data from approved semantic views.
- The **instructions** tell the agent how to interpret the returned evidence.
- The **ADK Runner** manages the execution and tool hand-off.

In [ ]:
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

INVENTORY_AGENT_INSTRUCTION = """
You are the Inventory Risk Analyst for Essex Central Distribution Centre
(warehouse WH-ESSEX-01).

Your task is to answer inventory-risk questions using only the supplied
read-only BigQuery tools.

Business definitions:
- Available quantity is on-hand quantity minus reserved quantity.
- Shortage units are max(safety stock minus available quantity, 0).
- Inventory value at risk is shortage units multiplied by unit cost in GBP.
- This is the cost value of the safety-stock gap, not lost revenue or realised loss.
- The workshop data contains one snapshot dated 2026-10-01; do not claim trends.

Operating rules:
1. Use get_inventory_risk for product-level evidence.
2. Use get_inventory_summary for portfolio-level totals.
3. Do not invent product names, quantities, values, dates or causes.
4. Keep monetary values in GBP and show units/denominators where relevant.
5. If a tool returns no evidence, say what is unavailable.
6. You are read-only. Never claim to update inventory, create orders, approve spend
   or contact suppliers.
7. Summarise the most important findings, cite product IDs/names and explain
   why the products are at risk using the returned fields.
"""

inventory_agent = Agent(
    name="inventory_risk_agent",
    model=MODEL_ID,
    description="Read-only inventory risk analyst for the Essex distribution centre.",
    instruction=INVENTORY_AGENT_INSTRUCTION,
    tools=[get_inventory_risk, get_inventory_summary],
)

inventory_app = App(
    name="gdg_essex_first_data_agent",
    root_agent=inventory_agent,
)
inventory_session_service = InMemorySessionService()
inventory_runner = Runner(
    app=inventory_app,
    session_service=inventory_session_service,
)

print("Agent created:", inventory_agent.name)

## 4. Ask the agent a real business question

The runner creates a session and streams events. The final answer is shown below. If the model decides it needs evidence, ADK invokes one of the Python tools and returns the tool result to the model before it responds.

In [ ]:
async def ask_inventory_agent(question: str, user_id: str = "gdg_essex_participant"):
    session = await inventory_session_service.create_session(
        app_name="gdg_essex_first_data_agent",
        user_id=user_id,
    )
    message = genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=question)],
    )

    final_parts = []
    event_log = []

    async for event in inventory_runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        event_log.append({
            "author": getattr(event, "author", None),
            "is_final": event.is_final_response(),
            "has_content": bool(getattr(event, "content", None)),
        })
        if event.is_final_response() and event.content and event.content.parts:
            final_parts.extend(
                part.text for part in event.content.parts
                if getattr(part, "text", None)
            )

    return {
        "answer": "\n".join(final_parts).strip(),
        "events": event_log,
        "session_id": session.id,
    }

QUESTION = (
    "Which products are most at risk of falling below safety stock? "
    "Give me the five highest inventory value-at-risk items, their shortage "
    "units and GBP value at risk, then summarise the overall inventory position. "
    "Explain what value at risk means in this dataset."
)

agent_result = await ask_inventory_agent(QUESTION)
print(agent_result["answer"])

## 5. Review the execution evidence

The agent answer is only useful if we can inspect how it was produced. The event summary below is a first look at the ADK execution lifecycle. In Notebook 3, we will build richer tracing and multi-agent observability.

In [ ]:
print("ADK event summary")
for index, event in enumerate(agent_result["events"], start=1):
    print(
        f"{index:02d}. author={event['author']!r} "
        f"final={event['is_final']} content={event['has_content']}"
    )

if not agent_result["answer"]:
    raise RuntimeError(
        "No final agent response was captured. Review the preceding ADK event output and traceback."
    )

## 6. Check the answer against canonical metrics

An agent's natural-language answer should not be treated as the source of truth. Compare its portfolio figures with the deterministic BigQuery summary. The summary view is the canonical source for the workshop's inventory KPIs.

In [ ]:
from decimal import Decimal

if not inventory_summary_evidence["rows"]:
    raise RuntimeError("The inventory summary view returned no rows.")

canonical = inventory_summary_evidence["rows"][0]
print("Canonical snapshot:", canonical["snapshot_date"])
print("Warehouse:", canonical["warehouse_id"])
print("Products below safety stock:", canonical["products_below_safety_stock"])
print("Shortage units:", canonical["shortage_units"])
print("Inventory value at risk: £{:,.2f}".format(
    float(canonical["inventory_value_at_risk_gbp"])
))

print("\nInterpretation check:")
print("• Does the agent describe the figure as a safety-stock gap rather than lost revenue?")
print("• Does it use the snapshot date rather than implying a live stock position?")
print("• Can you trace each product figure to the tool result above?")

## 7. Managed Conversational Analytics — comparison, not a blocker

Google Cloud's **Conversational Analytics** is an alternative managed approach for asking natural-language questions over governed analytical data. It is useful to compare the interaction model with the custom ADK agent you just built.

| Dimension | ADK agent in this notebook | Managed Conversational Analytics |
|---|---|---|
| What you build | Agent instructions, tools, runtime and session flow | A managed conversational analytics experience over configured data |
| Tool/control surface | Explicit Python functions and allow-listed BigQuery views | Managed data/semantic configuration and service capabilities |
| Custom workflow | You control the ADK agent and can extend it with specialists | Depends on the managed service's supported integration and orchestration features |
| Workshop status | Core hands-on path | Optional comparison / demo path |

**Important:** this notebook does not call or configure the managed Conversational Analytics service. Its exact API, permissions and integration behaviour must be verified for the workshop's delivery environment. The ADK path remains complete without it.

## 8. Core Exercise, Explore Further and Advanced Challenge

### Core Exercise — complete during the workshop

1. Run the question in Section 6.
2. Identify which tool(s) the agent needed.
3. Compare the portfolio metrics with the canonical summary in Section 8.
4. Check that the answer distinguishes inventory value at risk from lost revenue.

### Explore Further

Change the question to ask for:
- The highest shortage-unit products, irrespective of cost.
- A plain-English explanation of `CRITICAL`, `AT_RISK` and `HEALTHY`.
- A summary of how available quantity differs from on-hand quantity.

Use only evidence already exposed by the tools. If the tool surface does not expose a field needed to answer, explain that limitation rather than guessing.

### Advanced Challenge

Add a new read-only tool that answers a narrowly defined inventory question. Before registering it with the agent:

- Use a fixed SQL query against an approved semantic view.
- Parameterise any filter values.
- Enforce a row limit and maximum bytes billed.
- Return structured data with a clear schema.
- Add a unit test for its query construction and boundary behaviour.

Do not add arbitrary SQL execution or write operations.

## 9. Knowledge check

1. What is the difference between a model and an agent?
2. Why is a narrow Python tool safer than allowing the model to generate arbitrary SQL?
3. Which component manages the execution and tool hand-off in this notebook?
4. What does Harness Engineering cover?
5. Why is inventory value at risk not the same as lost revenue?
6. Why should the answer be checked against the canonical BigQuery summary?

### Recap

You built a first enterprise data agent with Gemini + ADK + read-only BigQuery tools. The tool surface is constrained, business definitions are explicit, and the answer can be checked against canonical metrics.

**Next:** Notebook 2 adds enterprise expertise: the Zetaris-inspired Data Harness pattern, semantic catalogue, reusable Agent Skills and three domain specialists.